# Step 4: Cross-Domain Evaluation and Visualization

This notebook covers:
1.  **Setup**: Mounting Drive and installing dependencies.
2.  **Dataset Preparation**: Extracting the Cityscapes validation set.
3.  **Model Loading**: Loading EoMT-Cityscapes (Semantic) and EoMT-COCO (Panoptic).
4.  **Visualization**: Side-by-side comparison of predictions on Cityscapes images.
5.  **Quantitative Evaluation**: mIoU comparison on Cityscapes validation set with class mapping.

## 1. Setup & Initialization

In [ ]:
from google.colab import drive
import os
import yaml
import torch
import torch.nn.functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import zipfile
from lightning import seed_everything

# Mount Drive
drive.mount('/content/drive')

# Change to project directory
project_path = '/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt'
os.chdir(project_path)
print(f"Current directory: {os.getcwd()}")

# Install dependencies
!pip install lightning gitignore_parser > /dev/null
!pip install -U jsonargparse[signatures]>=4.27.7 > /dev/null

seed_everything(0, verbose=False)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Dataset Preparation
Extract Cityscapes validation set from zip files.

In [ ]:
CITYSCAPES_DATA_PATH = "./data/cityscapes"
zips = [
    ("leftImg8bit_trainvaltest.zip", "leftImg8bit"),
    ("gtFine_trainvaltest.zip", "gtFine")
]

for zip_name, check_dir in zips:
    zip_path = os.path.join(CITYSCAPES_DATA_PATH, zip_name)
    target_check = os.path.join(CITYSCAPES_DATA_PATH, check_dir)

    if os.path.exists(zip_path) and not os.path.exists(target_check):
        print(f"Extracting {zip_name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(CITYSCAPES_DATA_PATH)
    else:
        print(f"{zip_name} already extracted or missing.")

print("Dataset preparation complete.")

## 3. Model Loading & Dataloaders

In [ ]:
from datasets.cityscapes_semantic import CityscapesSemantic
from datasets.coco_panoptic import COCOPanoptic
from models.vit import ViT
from models.eomt import EoMT
from training.mask_classification_semantic import MaskClassificationSemantic
from training.mask_classification_panoptic import MaskClassificationPanoptic

def load_eomt_model(config_path, ckpt_path, data_class, model_class, data_path, device):
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)

    # Data Module
    data = data_class(
        path=data_path,
        batch_size=1,
        num_workers=0,
        check_empty_targets=False,
        **cfg["data"].get("init_args", {})
    )

    img_size = getattr(data, "img_size", (640, 640))
    num_classes = getattr(data, "num_classes", 19)

    # Encoder
    encoder_cfg = cfg["model"]["init_args"]["network"]["init_args"]["encoder"]
    encoder = ViT(img_size=img_size, **encoder_cfg.get("init_args", {}))

    # Network
    network_cfg = cfg["model"]["init_args"]["network"]
    network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
    network = EoMT(
        masked_attn_enabled=False,
        num_classes=num_classes,
        encoder=encoder,
        **network_kwargs,
    )

    # Lightning Module
    model_kwargs = {k: v for k, v in cfg["model"]["init_args"].items() if k != "network"}
    if "stuff_classes" in cfg["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = cfg["data"]["init_args"]["stuff_classes"]

    model = model_class(
        img_size=img_size,
        num_classes=num_classes,
        network=network,
        **model_kwargs,
    ).eval().to(device)

    # Load Weights
    if os.path.exists(ckpt_path):
        state_dict = torch.load(ckpt_path, map_location=device, weights_only=True)
        model.load_state_dict(state_dict, strict=False)
        print(f"Loaded weights from {ckpt_path}")
    else:
        print(f"Warning: {ckpt_path} not found.")

    return model, data

print("Loading Cityscapes Model...")
model_cs, data_cs = load_eomt_model(
    "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml",
    "eomt_cityscapes.bin",
    CityscapesSemantic,
    MaskClassificationSemantic,
    "./data/cityscapes",
    device
)

print("\nLoading COCO Model...")
model_coco, data_coco = load_eomt_model(
    "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml",
    "eomt_coco.bin",
    COCOPanoptic,
    MaskClassificationPanoptic,
    "./data",
    device
)

print("\nInitializing Cityscapes Dataloader...")
data_cs.setup()
val_loader = data_cs.val_dataloader()

## 4. Visualization
Visualize predictions from both models on a sample Cityscapes image.

In [ ]:
def create_mapping(images, ignore_index=255):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array([plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))])
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping

def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

def draw_borders(sem, inst, mapping):
    h, w = sem.shape
    out = apply_colormap(sem, mapping)
    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out

img_idx = 0
img, target = val_loader.dataset[img_idx]

with torch.no_grad(), autocast(device_type="cuda", dtype=torch.float16):
    # Cityscapes Inference
    # Do not resize manually! window_imgs_semantic handles full-size images via sliding windows.
    cpu_uint8_imgs = [img.cpu().to(torch.uint8)]
    crops, origins = model_cs.window_imgs_semantic(cpu_uint8_imgs)
    crops = crops.to(device)
    mask_logits_per_layer, class_logits_per_layer = model_cs(crops)
    mask_logits = F.interpolate(mask_logits_per_layer[-1], model_cs.img_size, mode="bilinear")
    crop_logits = model_cs.to_per_pixel_logits_semantic(mask_logits, class_logits_per_layer[-1])
    logits_cs_full = model_cs.revert_window_logits_semantic(crop_logits, origins, [img.shape[-2:]])[0]
    pred_cs = logits_cs_full.argmax(0).cpu().numpy()

    # COCO Inference
    device_uint8_imgs = [img.to(device).to(torch.uint8)]
    transformed_imgs = model_coco.resize_and_pad_imgs_instance_panoptic(device_uint8_imgs)
    m_logits_p, c_logits_p = model_coco(transformed_imgs)
    m_logits = F.interpolate(m_logits_p[-1], model_coco.img_size, mode="bilinear")
    m_logits = model_coco.revert_resize_and_pad_logits_instance_panoptic(m_logits, [img.shape[-2:]])
    pred_coco_pan = model_coco.to_per_pixel_preds_panoptic(
        m_logits, c_logits_p[-1], model_coco.stuff_classes, model_coco.mask_thresh, model_coco.overlap_thresh
    )[0].cpu().numpy()
    sem_coco, inst_coco = pred_coco_pan[..., 0], pred_coco_pan[..., 1]

all_sem_ids = np.unique(np.concatenate([pred_cs.flatten(), sem_coco.flatten()]))
mapping = {sid: plt.cm.hsv(i/len(all_sem_ids))[:3] for i, sid in enumerate(all_sem_ids)}
mapping[255] = [0, 0, 0]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img.permute(1, 2, 0).numpy())
axes[0].set_title("Original Image")
axes[1].imshow(apply_colormap(pred_cs, mapping))
axes[1].set_title("Cityscapes Semantic Prediction")
axes[2].imshow(draw_borders(sem_coco, inst_coco, mapping))
axes[2].set_title("COCO Panoptic Prediction")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## Quantitative evaluation

In [ ]:
import gc
import torch

# List of heavy variables from the visualization step that might be holding GPU memory
vars_to_delete = [
    'model_cs', 'model_coco', 'data_cs', 'data_coco', 'val_loader',
    'img', 'target', 'crops', 'mask_logits_per_layer', 'class_logits_per_layer',
    'mask_logits', 'crop_logits', 'logits_cs_full', 'm_logits_p', 'c_logits_p',
    'm_logits', 'pred_coco_pan', 'device_uint8_imgs', 'transformed_imgs', 'cpu_uint8_imgs'
]

# Delete them from globals if they exist
for var in vars_to_delete:
    if var in globals():
        del globals()[var]

# Force garbage collection and empty the CUDA cache completely
gc.collect()
torch.cuda.empty_cache()

print(f"Allocated GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved GPU memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print("Notebook GPU memory aggressively freed!")

In [ ]:
import torch

# 1. Convert the raw weights into a valid PyTorch Lightning checkpoint
raw_weights = torch.load("eomt_cityscapes.bin", map_location="cpu")
pl_ckpt = {
    "state_dict": raw_weights,
    "pytorch-lightning_version": "2.0.0" # Mock version to satisfy the CLI
}
torch.save(pl_ckpt, "eomt_cityscapes_pl.ckpt")
print("Converted checkpoint saved as eomt_cityscapes_pl.ckpt")

# 2. Run the CLI validation using the new formatted checkpoint
# Added PYTORCH_ALLOC_CONF and forced batch_size=1 to avoid OOM
!PYTORCH_ALLOC_CONF=expandable_segments:True python main.py validate \
    --config configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
    --ckpt_path eomt_cityscapes_pl.ckpt \
    --data.init_args.path ./data/cityscapes \
    --data.init_args.batch_size 1


In [ ]:
import os
import json
import pandas as pd
import torch
import wandb
import numpy as np
from tqdm import tqdm
import torch.nn.functional as F
from torch.amp.autocast_mode import autocast
from google.colab import userdata
from datasets.cityscapes_semantic import CityscapesSemantic
from datasets.coco_panoptic import COCOPanoptic

# 1. Set WandB API Key
try:
    os.environ["WANDB_API_KEY"] = userdata.get('WANDDB-API-KEY')
    wandb.init(project="eomt-cross-domain", name="coco-on-cityscapes")
except userdata.SecretNotFoundError:
    print("WANDDB-API-KEY not found in secrets. Please add it to the 🔑 menu.")

# 2. Reload COCO Model correctly using its native data class
print("\nReloading COCO Model...")
model_coco, _ = load_eomt_model(
    "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml",
    "eomt_coco.bin",
    COCOPanoptic,
    MaskClassificationPanoptic,
    "./data",
    device
)

# 3. Initialize Cityscapes dataloader independently
print("Initializing Cityscapes Dataloader...")
data_cs = CityscapesSemantic(
    path="./data/cityscapes",
    batch_size=1,
    num_workers=0
)
data_cs.setup()
val_loader = data_cs.val_dataloader()

# 4. Load the specific mapping generated by map_coco_classes.py
mapping_file = "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt/coco-classes-mapping/coco_mapping_80to91.json"
coco_to_cs_map = {}
if os.path.exists(mapping_file):
    with open(mapping_file, 'r') as file:
        data = json.load(file)
        # Ensure keys and values are integers for tensor mapping
        coco_to_cs_map = {int(k): int(v) for k, v in data.items()}
    print(f"\nSuccessfully loaded mapping from: {mapping_file}")
else:
    print(f"\nWarning: Mapping file {mapping_file} not found! Did you run map_coco_classes.py?")

print(f"Loaded {len(coco_to_cs_map)} class mappings.")

def map_classes(prediction_tensor, mapping_dict, ignore_idx=255):
    mapped = torch.full_like(prediction_tensor, ignore_idx)
    for coco_id, cs_id in mapping_dict.items():
        mapped[prediction_tensor == coco_id] = cs_id
    return mapped

# Helper for mIoU calculation
def fast_hist(a, b, n):
    k = (a >= 0) & (a < n)
    return np.bincount(n * a[k].astype(int) + b[k].astype(int), minlength=n ** 2).reshape(n, n)

# 5. Custom Evaluation Loop
print("\nStarting cross-domain evaluation...")
model_coco.eval()
# Increased to 256 to prevent IndexError since the new mapping outputs IDs up to 91 (or higher depending on mapping)
num_classes_hist = 256
hist = np.zeros((num_classes_hist, num_classes_hist))

with torch.no_grad(), autocast(device_type="cuda", dtype=torch.float16):
    for batch in tqdm(val_loader, desc="Evaluating COCO on Cityscapes"):
        imgs, targets = batch
        imgs = [img.to(device).to(torch.uint8) for img in imgs]

        # Convert Cityscapes target dictionaries to a dense semantic map
        target_dict = targets[0]
        semantic_gt = torch.full((imgs[0].shape[-2], imgs[0].shape[-1]), 255, device=device)
        if 'masks' in target_dict and 'labels' in target_dict:
            for mask, label in zip(target_dict['masks'], target_dict['labels']):
                semantic_gt[mask] = label

        # 1. Get COCO Predictions
        transformed_imgs = model_coco.resize_and_pad_imgs_instance_panoptic(imgs)
        m_logits_p, c_logits_p = model_coco(transformed_imgs)
        m_logits = F.interpolate(m_logits_p[-1], model_coco.img_size, mode="bilinear")
        m_logits = model_coco.revert_resize_and_pad_logits_instance_panoptic(m_logits, [imgs[0].shape[-2:]])

        pred_coco_pan = model_coco.to_per_pixel_preds_panoptic(
            m_logits, c_logits_p[-1], model_coco.stuff_classes,
            model_coco.mask_thresh, model_coco.overlap_thresh
        )[0]

        sem_coco = pred_coco_pan[..., 0] # Extract semantic map

        # 2. Apply Mapping
        mapped_sem_coco = map_classes(sem_coco, coco_to_cs_map, ignore_idx=255)

        # 3. Calculate Confusion Matrix for mIoU
        pred_np = mapped_sem_coco.cpu().numpy().flatten()
        target_np = semantic_gt.cpu().numpy().flatten()

        hist += fast_hist(target_np, pred_np, num_classes_hist)

# Calculate final mIoU
ious = np.diag(hist) / (hist.sum(axis=1) + hist.sum(axis=0) - np.diag(hist) + 1e-8)
valid_classes = hist.sum(axis=1) > 0
miou = np.nanmean(ious[valid_classes]) * 100

print(f"\nEvaluation completed over the full dataset!")
print(f"Estimated mIoU (Cross-Domain): {miou:.2f}%")
